In [1]:
!pip install rapidfuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 51.1 MB/s eta 0:00:0000:01


In [2]:
import numpy as np
import pandas as pd

## 1. Load & Inspect Data

In [3]:
df = pd.read_csv('/kaggle/input/datasets/syedanwarafridi/vehicle-sales-data/car_prices.csv')
df.head()

,year,make,model,trim,body,transmission,vin,state,condition,odometer,color,interior,seller,mmr,sellingprice,saledate
0,2015,Kia,Sorento,LX,SUV,automatic,5xyktca69fg566472,ca,5.0,16639.0,white,black,kia motors america inc,20500.0,21500.0,Tue Dec 16 2014 12:30:00 GMT-0800 (PST)
1,2015,Kia,Sorento,LX,SUV,automatic,5xyktca69fg561319,ca,5.0,9393.0,white,beige,kia motors america inc,20800.0,21500.0,Tue Dec 16 2014 12:30:00 GMT-0800 (PST)
2,2014,BMW,3 Series,328i SULEV,Sedan,automatic,wba3c1c51ek116351,ca,45.0,1331.0,gray,black,financial services remarketing (lease),31900.0,30000.0,Thu Jan 15 2015 04:30:00 GMT-0800 (PST)
3,2015,Volvo,S60,T5,Sedan,automatic,yv1612tb4f1310987,ca,41.0,14282.0,white,black,volvo na rep/world omni,27500.0,27750.0,Thu Jan 29 2015 04:30:00 GMT-0800 (PST)
4,2014,BMW,6 Series Gran Coupe,650i,Sedan,automatic,wba6b2c57ed129731,ca,43.0,2641.0,gray,black,financial services remarketing (lease),66000.0,67000.0,Thu Dec 18 2014 12:30:00 GMT-0800 (PST)


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 558837 entries, 0 to 558836
Data columns (total 16 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   year          558837 non-null  int64  
 1   make          548536 non-null  object 
 2   model         548438 non-null  object 
 3   trim          548186 non-null  object 
 4   body          545642 non-null  object 
 5   transmission  493485 non-null  object 
 6   vin           558833 non-null  object 
 7   state         558837 non-null  object 
 8   condition     547017 non-null  float64
 9   odometer      558743 non-null  float64
 10  color         558088 non-null  object 
 11  interior      558088 non-null  object 
 12  seller        558837 non-null  object 
 13  mmr           558799 non-null  float64
 14  sellingprice  558825 non-null  float64
 15  saledate      558825 non-null  object 
dtypes: float64(4), int64(1), object(11)
memory usage: 68.2+ MB


In [5]:
df.describe()

,year,condition,odometer,mmr,sellingprice
count,558837.000000,547017.000000,558743.000000,558799.000000,558825.000000
mean,2010.038927,30.672365,68320.017767,13769.377495,13611.358810
std,3.966864,13.402832,53398.542821,9679.967174,9749.501628
min,1982.000000,1.000000,1.000000,25.000000,1.000000
25%,2007.000000,23.000000,28371.000000,7100.000000,6900.000000
50%,2012.000000,35.000000,52254.000000,12250.000000,12100.000000
75%,2013.000000,42.000000,99109.000000,18300.000000,18200.000000
max,2015.000000,49.000000,999999.000000,182000.000000,230000.000000


## 2. Handle Missing Values & Filter Invalid Rows

In [6]:
# Safety check: convert whitespace-only strings to real NaN
before_na = df.isna().sum().sum()
df = df.replace(r'^\s*$', np.nan, regex=True)
after_na = df.isna().sum().sum()
print(after_na - before_na, "empty/whitespace-only cells converted to NaN")

0 empty/whitespace-only cells converted to NaN


**Result: 0 conversions** -- pandas' read_csv already converts empty
CSV fields to NaN on load, so no whitespace-masked values remain.

In [7]:
# Excluding vehicles older than 1995: very old cars are rare in this dataset
# and behave more like collector/vintage items than typical resale-market vehicles,
# which would skew the price/mileage analysis.

before = len(df)
df = df[df['year'] >= 1995]
print(before - len(df), "rows removed (year < 1995)")

913 rows removed (year < 1995)


In [8]:
# Dropping trim, interior, and seller -- not needed for this analysis,
# which focuses on make, body type, year, mileage, and selling price.
df.drop(['trim', 'interior', 'seller'], axis=1, inplace=True)

## 3. Standardize Text & Categorical Values

In [9]:
df = df.apply(lambda x: x.str.strip().str.lower() if x.dtype == "object" else x)

In [10]:
df.head()

,year,make,model,body,transmission,vin,state,condition,odometer,color,mmr,sellingprice,saledate
0,2015,kia,sorento,suv,automatic,5xyktca69fg566472,ca,5.0,16639.0,white,20500.0,21500.0,tue dec 16 2014 12:30:00 gmt-0800 (pst)
1,2015,kia,sorento,suv,automatic,5xyktca69fg561319,ca,5.0,9393.0,white,20800.0,21500.0,tue dec 16 2014 12:30:00 gmt-0800 (pst)
2,2014,bmw,3 series,sedan,automatic,wba3c1c51ek116351,ca,45.0,1331.0,gray,31900.0,30000.0,thu jan 15 2015 04:30:00 gmt-0800 (pst)
3,2015,volvo,s60,sedan,automatic,yv1612tb4f1310987,ca,41.0,14282.0,white,27500.0,27750.0,thu jan 29 2015 04:30:00 gmt-0800 (pst)
4,2014,bmw,6 series gran coupe,sedan,automatic,wba6b2c57ed129731,ca,43.0,2641.0,gray,66000.0,67000.0,thu dec 18 2014 12:30:00 gmt-0800 (pst)


### Parse Sale Date

In [11]:
df['saledate'] = df['saledate'].str[4:-24]

In [12]:
df['saledate'] = pd.to_datetime(df['saledate'], format='%b %d %Y')

### Standardize Brand Names (fuzzy matching)

In [13]:
df['make'].unique()

array(['kia', 'bmw', 'volvo', 'nissan', 'chevrolet', 'audi', 'ford',
       'hyundai', 'buick', 'cadillac', 'acura', 'lexus', 'infiniti',
       'jeep', 'mercedes-benz', 'mitsubishi', 'mazda', 'mini',
       'land rover', 'lincoln', 'jaguar', 'volkswagen', 'toyota',
       'subaru', 'scion', 'porsche', nan, 'dodge', 'fiat', 'chrysler',
       'ferrari', 'honda', 'gmc', 'ram', 'smart', 'bentley', 'pontiac',
       'saturn', 'maserati', 'mercury', 'hummer', 'landrover', 'mercedes',
       'gmc truck', 'saab', 'suzuki', 'oldsmobile', 'isuzu', 'dodge tk',
       'rolls-royce', 'mazda tk', 'hyundai tk', 'mercedes-b', 'vw',
       'daewoo', 'chev truck', 'ford tk', 'plymouth', 'ford truck',
       'tesla', 'airstream', 'dot', 'aston martin', 'geo', 'fisker',
       'lamborghini', 'lotus'], dtype=object)

In [14]:
from rapidfuzz import process

makes = df['make'].dropna().unique()

# find all similar pairs
for make in makes:
    matches = process.extract(make, makes, score_cutoff=80) 
    if len(matches) > 1:
        print(make, "->", matches)

ford -> [('ford', 100.0, 6), ('ford tk', 90.0, 55), ('ford truck', 90.0, 57)]
hyundai -> [('hyundai', 100.0, 7), ('hyundai tk', 95.0, 50)]
mercedes-benz -> [('mercedes-benz', 100.0, 14), ('mercedes', 90.0, 41), ('mercedes-b', 86.95652173913044, 51)]
mazda -> [('mazda', 100.0, 16), ('mazda tk', 90.0, 49)]
land rover -> [('land rover', 100.0, 18), ('landrover', 94.73684210526316, 40)]
dodge -> [('dodge', 100.0, 26), ('dodge tk', 90.0, 47)]
gmc -> [('gmc', 100.0, 31), ('gmc truck', 90.0, 42)]
landrover -> [('landrover', 100.0, 40), ('land rover', 94.73684210526316, 18)]
mercedes -> [('mercedes', 100.0, 41), ('mercedes-benz', 90.0, 14), ('mercedes-b', 88.88888888888889, 51)]
gmc truck -> [('gmc truck', 100.0, 42), ('gmc', 90.0, 31)]
dodge tk -> [('dodge tk', 100.0, 47), ('dodge', 90.0, 26)]
mazda tk -> [('mazda tk', 100.0, 49), ('mazda', 90.0, 16)]
hyundai tk -> [('hyundai tk', 100.0, 50), ('hyundai', 95.0, 7)]
mercedes-b -> [('mercedes-b', 100.0, 51), ('mercedes', 88.88888888888889, 41), 

In [15]:
replace_dict = {
    'ford tk': 'ford',
    'ford truck': 'ford',
    'hyundai tk': 'hyundai',
    'mercedes': 'mercedes-benz',
    'mercedes-b': 'mercedes-benz',
    'mazda tk': 'mazda',
    'landrover': 'land rover',
    'dodge tk': 'dodge',
    'gmc truck': 'gmc',
    'vw': 'volkswagen'
}

df['make'] = df['make'].replace(replace_dict)


In [16]:
df['make'].unique()

array(['kia', 'bmw', 'volvo', 'nissan', 'chevrolet', 'audi', 'ford',
       'hyundai', 'buick', 'cadillac', 'acura', 'lexus', 'infiniti',
       'jeep', 'mercedes-benz', 'mitsubishi', 'mazda', 'mini',
       'land rover', 'lincoln', 'jaguar', 'volkswagen', 'toyota',
       'subaru', 'scion', 'porsche', nan, 'dodge', 'fiat', 'chrysler',
       'ferrari', 'honda', 'gmc', 'ram', 'smart', 'bentley', 'pontiac',
       'saturn', 'maserati', 'mercury', 'hummer', 'saab', 'suzuki',
       'oldsmobile', 'isuzu', 'rolls-royce', 'daewoo', 'chev truck',
       'plymouth', 'tesla', 'airstream', 'dot', 'aston martin', 'geo',
       'fisker', 'lamborghini', 'lotus'], dtype=object)

### Standardize Body Types

In [17]:
df['body'].unique()

array(['suv', 'sedan', 'convertible', 'coupe', 'wagon', 'hatchback',
       'crew cab', 'g coupe', 'g sedan', 'elantra coupe', 'genesis coupe',
       'minivan', nan, 'van', 'double cab', 'crewmax cab', 'access cab',
       'king cab', 'supercrew', 'cts coupe', 'extended cab',
       'e-series van', 'supercab', 'regular cab', 'g convertible', 'koup',
       'quad cab', 'cts-v coupe', 'g37 convertible', 'club cab',
       'xtracab', 'q60 convertible', 'cts wagon', 'g37 coupe', 'mega cab',
       'cab plus 4', 'q60 coupe', 'cab plus', 'beetle convertible',
       'tsx sport wagon', 'promaster cargo van',
       'granturismo convertible', 'cts-v wagon', 'ram van', 'transit van',
       'navitgation', 'regular-cab'], dtype=object)

In [18]:
# NOTE: cargo vans (promaster cargo van, transit van, ram van) are grouped into
# 'minivan' here for simplicity. Consider a separate 'van' category if van vs.
# minivan pricing needs to be distinguished later.
replace_dict = {
    'g coupe': 'coupe',
    'g sedan': 'sedan',
    'elantra coupe': 'coupe',
    'genesis coupe': 'coupe',
    'van': 'minivan',
    'supercrew': 'crew cab',
    'double cab': 'crew cab',
    'crewmax cab': 'crew cab',
    'access cab': 'crew cab',
    'king cab': 'crew cab',
    'cts coupe': 'coupe',
    'extended cab': 'crew cab',
    'e-series van': 'minivan',
    'supercab': 'crew cab',
    'regular cab': 'crew cab',
    'g convertible': 'convertible',
    'cts coupe': 'coupe',
    'quad cab': 'crew cab',
    'cts-v coupe': 'coupe',
    'g37 convertible': 'convertible',
    'xtracab': 'crew cab',
    'q60 convertible': 'convertible',
    'cts wagon': 'wagon',
    'g37 coupe': 'coupe',
    'cab plus 4': 'crew cab',
    'q60 coupe': 'coupe',
    'cab plus': 'crew cab',
    'beetle convertible': 'convertible',
    'tsx sport wagon': 'wagon',
    'promaster cargo van': 'minivan',
    'granturismo convertible': 'convertible',
    'cts-v wagon': 'wagon',
    'ram van': 'minivan',
    'transit van': 'minivan',
    'regular-cab': 'crew cab'
}

df['body'] = df['body'].replace(replace_dict)


### Clean Transmission, State, Color

In [19]:
df['transmission'].unique()

array(['automatic', nan, 'manual', 'sedan'], dtype=object)

In [20]:
# 'sedan' showed up as a transmission value by data-entry error -- drop those rows

before = len(df)
df = df[df['transmission'] != 'sedan']
print(before - len(df), "rows removed (invalid transmission value)")

26 rows removed (invalid transmission value)


In [21]:
df['state'].unique()

array(['ca', 'tx', 'pa', 'mn', 'az', 'wi', 'tn', 'md', 'fl', 'ne', 'nj',
       'nv', 'oh', 'mi', 'ga', 'va', 'sc', 'nc', 'in', 'il', 'co', 'ut',
       'mo', 'ny', 'ma', 'pr', 'or', 'la', 'wa', 'hi', 'qc', 'ab', 'on',
       'ok', 'ms', 'nm', 'al', 'ns'], dtype=object)

In [22]:
df['color'].unique()

array(['white', 'gray', 'black', 'red', 'silver', 'blue', 'brown',
       'beige', 'purple', 'burgundy', '—', 'gold', 'yellow', 'green',
       'charcoal', nan, 'orange', 'off-white', 'turquoise', 'pink',
       'lime'], dtype=object)

In [23]:
df['color'] = df['color'].replace({
    'off-white': 'white',
    '—': np.nan
})

## 4. Remove Duplicates & Handle Remaining Missing Values

In [24]:
df.describe()

,year,condition,odometer,mmr,sellingprice,saledate
count,557898.000000,546168.000000,557804.000000,557886.000000,557886.000000,557886
mean,2010.067365,30.717193,68174.338839,13790.109494,13631.909677,2015-03-06 04:15:34.332820736
min,1995.000000,1.000000,1.000000,25.000000,1.000000,2014-01-01 00:00:00
25%,2007.000000,24.000000,28345.000000,7150.000000,6900.000000,2015-01-21 00:00:00
50%,2012.000000,35.000000,52168.000000,12300.000000,12100.000000,2015-02-13 00:00:00
75%,2013.000000,42.000000,98935.000000,18350.000000,18200.000000,2015-05-22 00:00:00
max,2015.000000,49.000000,999999.000000,182000.000000,230000.000000,2015-07-21 00:00:00
std,3.905307,13.364863,53212.704978,9674.169078,9744.225843,NaN


In [25]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 557898 entries, 0 to 558836
Data columns (total 13 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   year          557898 non-null  int64         
 1   make          547730 non-null  object        
 2   model         547632 non-null  object        
 3   body          544880 non-null  object        
 4   transmission  492667 non-null  object        
 5   vin           557898 non-null  object        
 6   state         557898 non-null  object        
 7   condition     546168 non-null  float64       
 8   odometer      557804 non-null  float64       
 9   color         532476 non-null  object        
 10  mmr           557886 non-null  float64       
 11  sellingprice  557886 non-null  float64       
 12  saledate      557886 non-null  datetime64[ns]
dtypes: datetime64[ns](1), float64(4), int64(1), object(7)
memory usage: 59.6+ MB


In [26]:
initial_rows = len(df)
df = df.dropna(subset=['vin', 'make', 'odometer'])
print(initial_rows - len(df), "rows with missing critical values have been removed")

10258 rows with missing critical values have been removed


In [27]:
initial_count = len(df)
df = df.drop_duplicates(subset=['vin', 'saledate'])
print(initial_count - len(df), "duplicates have been deleted")

98 duplicates have been deleted


In [28]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 547542 entries, 0 to 558836
Data columns (total 13 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   year          547542 non-null  int64         
 1   make          547542 non-null  object        
 2   model         547444 non-null  object        
 3   body          544697 non-null  object        
 4   transmission  484107 non-null  object        
 5   vin           547542 non-null  object        
 6   state         547542 non-null  object        
 7   condition     535965 non-null  float64       
 8   odometer      547542 non-null  float64       
 9   color         522341 non-null  object        
 10  mmr           547530 non-null  float64       
 11  sellingprice  547530 non-null  float64       
 12  saledate      547530 non-null  datetime64[ns]
dtypes: datetime64[ns](1), float64(4), int64(1), object(7)
memory usage: 58.5+ MB


## 5. Handle Outliers

The `describe()` output in Section 1 shows implausible extreme values: `odometer` max is 999,999 miles, which is not physically realistic for vehicles sold in 2014-2015. Investigate and address before exporting, rather than leaving it in silently.

In [29]:
# Check how many rows have an implausible odometer reading
suspicious_odometer = (df['odometer'] > 300000).sum()
print(suspicious_odometer, "rows with odometer above 300,000 miles")

659 rows with odometer above 300,000 miles


**Decision:** left as-is in this pass. Mileage above 300K is plausible for
commercial/fleet vehicles, not necessarily an error — unlike the $230K Ford
Escape, there's no single value here that stands out as impossible (999,999
was seen in raw `describe()`, but not confirmed as the majority case).
Filtering by mileage range is handled visually in the Tableau dashboard
instead of dropped from the source data here.

In [30]:
# Selling price also has a high max (~$230,000) from Section 1's describe().
# Sort by price descending so we see ALL high-value listings, not a partial
# unsorted sample -- genuine ultra-luxury cars (Bugatti, McLaren, Rolls-Royce
# Phantom, etc.) can legitimately sell above $200K, so we need the full
# picture before deciding what to remove.
pd.set_option('display.max_rows', 50)
df[df['sellingprice'] > 150000].sort_values('sellingprice', ascending=False)[['make', 'model', 'year', 'sellingprice']]

,make,model,year,sellingprice
344905,ford,escape,2014,230000.0
548169,ferrari,458 italia,2011,183000.0
446949,mercedes-benz,s-class,2015,173000.0
545523,rolls-royce,ghost,2013,171500.0
125095,rolls-royce,ghost,2012,169500.0
557570,rolls-royce,ghost,2012,169000.0
538347,rolls-royce,ghost,2012,167000.0
283534,bmw,i8,2014,165000.0
146917,bmw,i8,2014,165000.0
299198,bentley,continental gtc,2013,163000.0


Row `344905` (Ford Escape, 2014, \$230,000) doesn't match the price range
expected for this model. It's priced **~\$47,000 above** the next most
expensive vehicle in the dataset (Ferrari 458 Italia, \$183,000), while every
other high-value listing is a genuine luxury/exotic brand (Rolls-Royce,
Bentley, BMW i8, Porsche).

This strongly suggests a **data-entry error** rather than a real sale, so it
was removed by index. The impact on the overall analysis is minimal — a
single row out of ~547,000.

In [31]:
before = len(df)
df = df.drop(index=344905)
print(before - len(df), "row(s) removed due to implausible selling price")

1 row(s) removed due to implausible selling price


In [32]:
print("Final dataset:", len(df), "rows,", df.shape[1], "columns")

Final dataset: 547541 rows, 13 columns


## 6. Export Cleaned Data

In [33]:
df.to_csv('cleaned_car_sales_data.csv', index=False)

## Summary

| Step | Rows removed | Rows remaining |
|---|---|---|
| Initial load | -- | 558,837 |
| Filter year >= 1995 | 913 | 557,924 |
| Remove invalid transmission value | 26 | 557,898 |
| Remove rows missing vin/make/odometer | 10,258 | 547,640 |
| Remove duplicate (vin, saledate) | 98 | 547,542 |
| Remove implausible selling price (Ford Escape) | 1 | 547,541 |
| **Final dataset** | | **547,541**|

Also standardized:
- ~10 duplicate brand names (e.g. `ford tk`, `ford truck` -> `ford`) via fuzzy matching
- ~30 duplicate/overlapping body-type labels (e.g. `elantra coupe`, `genesis coupe` -> `coupe`)

**Note:** odometer readings above 300,000 miles were kept in this dataset --
they're plausible for commercial/fleet vehicles rather than clear errors.
Filtering by mileage range is applied visually in the Tableau dashboard
instead.

Exported to `cleaned_car_sales_data.csv`.